# 0-1 Knapsack Problem — Formulation and Solution (Python)> **ORKit Free** · Integer Programming (IP)---## Problem DescriptionGiven $n$ items, each with a **weight** $w_i$ and a **value** $v_i$, and a knapsackof capacity $W$, select items to **maximize total value** without exceeding capacity.---## Mathematical Formulation$$x_i \in \{0,1\}: \text{1 if item } i \text{ is selected}$$$$\max \sum_{i} v_i x_i$$$$\text{s.t.} \quad \sum_{i} w_i x_i \leq W$$

In [ ]:
import jsonimport pyomo.environ as pyo

## 1. Load Instance

In [ ]:
with open("../instances/small_5.json") as f:    data = json.load(f)items = data["items"]W = data["capacity"]print(f"Capacity: {W}, Items: {len(items)}")for it in items:    print(f"  Item {it['id']}: value={it['value']}, weight={it['weight']}")

## 2. Build the Model

In [ ]:
I = [it["id"] for it in items]v = {it["id"]: it["value"] for it in items}w = {it["id"]: it["weight"] for it in items}model = pyo.ConcreteModel("Knapsack01")model.I = pyo.Set(initialize=I)model.x = pyo.Var(model.I, within=pyo.Binary, doc="1 if item i is selected")model.obj = pyo.Objective(    expr=sum(v[i] * model.x[i] for i in model.I),    sense=pyo.maximize)model.capacity = pyo.Constraint(    expr=sum(w[i] * model.x[i] for i in model.I) <= W)

## 3. Solve

In [ ]:
solver = pyo.SolverFactory("appsi_highs")result = solver.solve(model, tee=False)print(f"Status: {result.solver.termination_condition}")print(f"Value : {pyo.value(model.obj):.0f}")

## 4. Selected Items

In [ ]:
selected = [i for i in I if pyo.value(model.x[i]) > 0.5]total_w = sum(w[i] for i in selected)total_v = sum(v[i] for i in selected)print(f"Selected: {selected}")print(f"Total weight: {total_w} / {W}")print(f"Total value : {total_v}")for i in selected:    print(f"  Item {i}: value={v[i]}, weight={w[i]}")

## Key Takeaways- Classic NP-hard combinatorial optimization problem- Dynamic programming solves it in pseudo-polynomial $O(nW)$ time- MIP approach generalizes to variants (multiple knapsacks, bounded, etc.)- Reference: Kellerer, Pferschy & Pisinger (2004)